# Missing indicator, random sample imputation, and choosing the imputer automatically
Rebuilt for pandas 3 and scikit-learn 1.9.
Data: `data/titanic.csv` (Kaggle's Titanic training file, 891 passengers) and `data/house_prices.csv`
(three columns of Kaggle's *House Prices: Advanced Regression Techniques* training file; the full file has 81 columns).
Every random draw has a fixed seed, so the numbers match the Note. Every imputer learns from the training set only.

In [1]:
# pandas for tables, Plotly for charts, scipy for the KDE curves, scikit-learn for the models
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## 1. Random sample imputation on a numerical column

In [2]:
# two inputs (Age, Fare) and the target (Survived)
df = pd.read_csv("data/titanic.csv", usecols=["Age", "Fare", "Survived"])
# share of missing values per column, in %
(df.isnull().mean() * 100).round(2)

Survived     0.00
Age         19.87
Fare         0.00
dtype: float64

In [3]:
# split first: the fill values may only come from the training set
X = df[["Age", "Fare"]]
y = df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
# copy() makes independent tables, so adding columns gives no SettingWithCopyWarning
X_train = X_train.copy()
X_test = X_test.copy()
print(X_train["Age"].isnull().sum(), "gaps in training,", X_test["Age"].isnull().sum(), "in test")

148 gaps in training, 29 in test


In [4]:
# the 'bag': every known Age of the training set
pool = X_train["Age"].dropna()

def random_fill(col, pool, seed):
    # replace each gap with a value drawn at random from the pool
    out = col.copy()
    gaps = out.isnull()
    # sample(n) draws n values; random_state fixes the draw
    out[gaps] = pool.sample(gaps.sum(), random_state=seed).values
    return out

# keep the original column next to the filled one, to compare them
X_train["Age_imputed"] = random_fill(X_train["Age"], pool, seed=42)
# the test set is filled from the TRAINING pool too
X_test["Age_imputed"] = random_fill(X_test["Age"], pool, seed=42)
X_train[X_train["Age"].isnull()].head()

,Age,Fare,Age_imputed
77,NaN,8.0500,22.0
868,NaN,9.5000,18.0
334,NaN,133.6500,38.0
295,NaN,27.7208,40.0
792,NaN,69.5500,52.0


In [5]:
# the variance barely changes
print("variance before:", round(X_train["Age"].var(), 2))
print("variance after: ", round(X_train["Age_imputed"].var(), 2))

variance before: 204.35
variance after:  200.03


In [6]:
# the covariance with Fare drops, because the drawn ages ignore Fare
X_train[["Fare", "Age", "Age_imputed"]].cov().round(2)

,Fare,Age,Age_imputed
Fare,2368.25,71.51,53.26
Age,71.51,204.35,204.35
Age_imputed,53.26,204.35,200.03


In [7]:
# density curves and box plots before and after
fig = make_subplots(1, 2, subplot_titles=["Density", "Box plot"])
grid = np.linspace(-10, 85, 400)
for col, colour in [("Age", "#4C78A8"), ("Age_imputed", "#F58518")]:
    curve = gaussian_kde(X_train[col].dropna())(grid)
    fig.add_trace(go.Scatter(x=grid, y=curve, name=col, line_color=colour), 1, 1)
    fig.add_trace(go.Box(y=X_train[col], name=col, marker_color=colour, showlegend=False), 1, 2)
fig.update_layout(template="simple_white", height=420)
fig.show()

## 2. Same input, same fill: one seed per row

In [8]:
def fill_one(row, pool):
    # a missing Age gets a random Age, seeded by this row's Fare
    if pd.isna(row["Age"]):
        return pool.sample(1, random_state=int(row["Fare"])).iloc[0]
    return row["Age"]

# a new passenger arrives with no Age, fare 56.4958
new = pd.Series({"Age": np.nan, "Fare": 56.4958})
# ask three times: the answer is the same each time
[fill_one(new, pool) for _ in range(3)]

[np.float64(34.0), np.float64(34.0), np.float64(34.0)]

In [9]:
# without a fixed seed, the same passenger gets a different Age each time
[pool.sample(1).iloc[0] for _ in range(3)]

[np.float64(2.0), np.float64(30.0), np.float64(28.0)]

## 3. Random sample imputation on categorical columns

In [10]:
house = pd.read_csv("data/house_prices.csv")
(house.isnull().mean() * 100).round(2)

GarageQual      5.55
FireplaceQu    47.26
SalePrice       0.00
dtype: float64

In [11]:
# the target stays inside the table here, to compare prices per category later
H_train, H_test = train_test_split(house, test_size=0.2, random_state=2)
H_train = H_train.copy()
H_test = H_test.copy()
for c in ["GarageQual", "FireplaceQu"]:
    pool_c = H_train[c].dropna()
    H_train[c + "_imputed"] = random_fill(H_train[c], pool_c, seed=42)
    H_test[c + "_imputed"] = random_fill(H_test[c], pool_c, seed=42)
H_train[["GarageQual", "FireplaceQu"]].isnull().sum()

GarageQual      65
FireplaceQu    557
dtype: int64

In [12]:
def shares(c):
    # each category's share among the rows that have a value, before and after
    before = H_train[c].value_counts(normalize=True)
    after = H_train[c + "_imputed"].value_counts(normalize=True)
    return (pd.concat([before, after], axis=1, keys=["before", "after"]) * 100).round(1)

shares("GarageQual")

,before,after
TA,95.1,95.2
Fa,3.7,3.7
Gd,1.0,0.9
Po,0.1,0.1
Ex,0.1,0.1


In [13]:
# even with 47% missing, the shares barely move: the draws follow the same proportions
shares("FireplaceQu")

,before,after
Gd,49.4,50.0
TA,41.2,40.7
Fa,4.1,4.0
Po,2.8,2.8
Ex,2.5,2.5


In [14]:
# but the mean sale price per category changes a lot for FireplaceQu
pd.concat([H_train.groupby("FireplaceQu")["SalePrice"].mean(),
           H_train.groupby("FireplaceQu_imputed")["SalePrice"].mean()],
          axis=1, keys=["before", "after"]).round(0)

,before,after
Ex,337145.0,240230.0
Fa,168994.0,161279.0
Gd,225668.0,184529.0
Po,135112.0,139591.0
TA,205749.0,175367.0


In [15]:
# houses with a gap sold for much less than the Gd or TA houses they were turned into
H_train.loc[H_train["FireplaceQu"].isnull(), "SalePrice"].mean().round(0)

np.float64(141182.0)

## 4. Missing indicator

In [16]:
# the same Titanic split, without the extra column of Section 1
X_train = X_train[["Age", "Fare"]]
X_test = X_test[["Age", "Fare"]]

# baseline: mean imputation, then logistic regression
si = SimpleImputer()  # strategy="mean" by default
X_train_trf = si.fit_transform(X_train)
X_test_trf = si.transform(X_test)
clf = LogisticRegression().fit(X_train_trf, y_train)
accuracy_score(y_test, clf.predict(X_test_trf))

0.6145251396648045

In [17]:
# MissingIndicator learns which columns had gaps in the training set
mi = MissingIndicator()
mi.fit(X_train)
mi.features_  # [0]: only column 0, Age

array([0])

In [18]:
# transform gives one True/False column per such column
X_train_missing = mi.transform(X_train)
X_test_missing = mi.transform(X_test)
X_train_missing[:5].ravel()

array([False, False, False, False, False])

In [19]:
# add the indicator as a third column, then impute and train again
X_train2 = X_train.assign(Age_NA=X_train_missing.ravel())
X_test2 = X_test.assign(Age_NA=X_test_missing.ravel())
si = SimpleImputer()
X_train_trf2 = si.fit_transform(X_train2)
X_test_trf2 = si.transform(X_test2)
clf = LogisticRegression().fit(X_train_trf2, y_train)
accuracy_score(y_test, clf.predict(X_test_trf2))

0.6312849162011173

In [20]:
# the same in one step: SimpleImputer with add_indicator=True
si = SimpleImputer(add_indicator=True)
X_train_trf3 = si.fit_transform(X_train)
X_test_trf3 = si.transform(X_test)
print(si.get_feature_names_out())
clf = LogisticRegression().fit(X_train_trf3, y_train)
accuracy_score(y_test, clf.predict(X_test_trf3))

['Age' 'Fare' 'missingindicator_Age']


0.6312849162011173

In [21]:
# why it helps here: passengers with no recorded Age survived less often
pd.Series({"Age missing": y_train[X_train["Age"].isnull()].mean(),
           "Age known": y_train[X_train["Age"].notnull()].mean()}).round(3)

Age missing    0.284
Age known      0.392
dtype: float64

In [22]:
# one split is a small sample: average the indicator's effect over 100 splits (5 folds x 20 repeats)
from sklearn.model_selection import cross_val_score, RepeatedStratifiedKFold
from sklearn.pipeline import make_pipeline
d = pd.read_csv("data/titanic.csv")
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=20, random_state=0)
for ai in [False, True]:
    p = make_pipeline(SimpleImputer(add_indicator=ai), LogisticRegression())
    print("add_indicator =", ai, round(cross_val_score(p, d[["Age", "Fare"]], d["Survived"], cv=cv).mean(), 4))

add_indicator = False 0.6566


add_indicator = True 0.6633


## 5. Choosing the imputer automatically with GridSearchCV

In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

df = pd.read_csv("data/titanic.csv")
# drop the columns we do not use
df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
X = df.drop(columns=["Survived"])
y = df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
X_train.isnull().sum()

Pclass        0
Sex           0
Age         148
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [24]:
# numerical columns: impute, then scale
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
# categorical columns: impute, then one-hot encode
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore")),
])
# each group of columns gets its own pipeline; other columns are dropped
preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_transformer, ["Age", "Fare"]),
    ("cat", categorical_transformer, ["Embarked", "Sex"]),
])
clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression()),
])
clf

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3I

In [25]:
# every setting has a name: step names joined by double underscores
[name for name in clf.get_params() if name.endswith("strategy") or name == "classifier__C"]

['preprocessor__num__imputer__strategy',
 'preprocessor__cat__imputer__strategy',
 'classifier__C']

In [26]:
param_grid = {
    "preprocessor__num__imputer__strategy": ["mean", "median"],
    "preprocessor__cat__imputer__strategy": ["most_frequent", "constant"],
    "classifier__C": [0.1, 1.0, 10, 100],
}
# 2 x 2 x 4 = 16 combinations, each scored by 10-fold cross-validation on the TRAINING set
grid_search = GridSearchCV(clf, param_grid, cv=10)
grid_search.fit(X_train, y_train)
print(grid_search.best_params_)
print(f"best cross-validation accuracy: {grid_search.best_score_:.3f}")

{'classifier__C': 1.0, 'preprocessor__cat__imputer__strategy': 'most_frequent', 'preprocessor__num__imputer__strategy': 'mean'}
best cross-validation accuracy: 0.788


In [27]:
# all 16 combinations, best first
cv_results = pd.DataFrame(grid_search.cv_results_)
cols = ["param_preprocessor__num__imputer__strategy", "param_preprocessor__cat__imputer__strategy",
        "param_classifier__C", "mean_test_score", "rank_test_score"]
cv_results.sort_values("rank_test_score")[cols].round(4)

,param_preprocessor__num__imputer__strategy,param_preprocessor__cat__imputer__strategy,param_classifier__C,mean_test_score,rank_test_score
7,median,constant,1.0,0.7879,1
6,mean,constant,1.0,0.7879,1
5,median,most_frequent,1.0,0.7879,1
4,mean,most_frequent,1.0,0.7879,1
11,median,constant,10.0,0.7879,1
10,mean,constant,10.0,0.7879,1
9,median,most_frequent,10.0,0.7879,1
8,mean,most_frequent,10.0,0.7879,1
12,mean,most_frequent,100.0,0.7879,1
13,median,most_frequent,100.0,0.7879,1


In [28]:
# the test set is used once, at the very end, on the refitted best pipeline
print(f"test accuracy: {grid_search.score(X_test, y_test):.3f}")

test accuracy: 0.760


In [29]:
# Why do the four imputer combinations tie? Change only how Age is filled, or drop Age (C = 1, same 10 folds)
from sklearn.model_selection import cross_val_score
def pipe(num_cols, age_imp):
    num = Pipeline(steps=[("imputer", age_imp), ("scaler", StandardScaler())])
    cat = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))])
    pre = ColumnTransformer(transformers=[("num", num, num_cols), ("cat", cat, ["Embarked", "Sex"])])
    return Pipeline(steps=[("preprocessor", pre), ("classifier", LogisticRegression(C=1))])
for name, cols_, imp in [("mean", ["Age", "Fare"], SimpleImputer()),
                         ("median", ["Age", "Fare"], SimpleImputer(strategy="median")),
                         ("constant 0", ["Age", "Fare"], SimpleImputer(strategy="constant", fill_value=0)),
                         ("constant 99", ["Age", "Fare"], SimpleImputer(strategy="constant", fill_value=99)),
                         ("no Age", ["Fare"], SimpleImputer())]:
    print(name, round(cross_val_score(pipe(cols_, imp), X_train, y_train, cv=10).mean(), 4))

mean 0.7879


median 0.7879


constant 0 0.7879


constant 99 0.7893


no Age 0.7879


## Grid search where the fill matters: the house prices

On the Titanic data every imputer ties, because `Age` barely drives the predictions. Here the column with gaps, `FireplaceQu`, drives the price, and its gaps mark cheaper houses.

In [30]:
from sklearn.linear_model import LinearRegression
house = pd.read_csv("data/house_prices.csv")
Xh = house[["GarageQual", "FireplaceQu"]]
yh = house["SalePrice"]
Xh_train, Xh_test, yh_train, yh_test = train_test_split(Xh, yh, test_size=0.2, random_state=2)
house_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore")),
    ("model", LinearRegression()),
])
# score = minus the mean absolute error in dollars, so higher is better
house_search = GridSearchCV(house_pipe, {"imputer__strategy": ["most_frequent", "constant"]},
                            cv=10, scoring="neg_mean_absolute_error")
house_search.fit(Xh_train, yh_train)
print(house_search.best_params_)
for s, score in zip(house_search.cv_results_["param_imputer__strategy"], house_search.cv_results_["mean_test_score"]):
    print(f"{s:14s} mean CV error: {-score:,.0f} dollars")
print(f"test error of the best pipeline: {-house_search.score(Xh_test, yh_test):,.0f} dollars")

{'imputer__strategy': 'constant'}
most_frequent  mean CV error: 53,447 dollars
constant       mean CV error: 46,382 dollars
test error of the best pipeline: 47,272 dollars


In [31]:
# the same comparison averaged over 100 splits (10 folds x 10 repeats) of all 1,460 houses
from sklearn.model_selection import RepeatedKFold
cv = RepeatedKFold(n_splits=10, n_repeats=10, random_state=0)
for s in ["most_frequent", "constant"]:
    err = -cross_val_score(house_pipe.set_params(imputer__strategy=s), Xh, yh, cv=cv,
                           scoring="neg_mean_absolute_error").mean()
    print(f"{s:14s} {err:,.0f} dollars")

most_frequent  54,015 dollars


constant       46,863 dollars
